In [2]:
import pandas as pd
import nltk
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

print("All libraries loaded successfully!")

All libraries loaded successfully!


In [3]:
# Load GoEmotions dataset from HuggingFace
dataset = load_dataset("go_emotions", "simplified")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})


In [4]:
# Convert train split to pandas DataFrame
df_train = dataset['train'].to_pandas()

# Look at first 5 rows
df_train.head()

,text,labels,id
0,My favourite food is anything I didn't have to...,[27],eebbqej
1,"Now if he does off himself, everyone will thin...",[27],ed00q6i
2,WHY THE FUCK IS BAYLESS ISOING,[2],eezlygj
3,To make her feel threatened,[14],ed7ypvh
4,Dirty Southern Wankers,[3],ed0bdzj


In [5]:
# Count rows with multiple labels
multiple_labels = df_train[df_train['labels'].apply(len) > 1]
print(f"Rows with multiple labels: {len(multiple_labels)}")
print(f"Total rows: {len(df_train)}")
print(f"Percentage: {len(multiple_labels)/len(df_train)*100:.2f}%")

Rows with multiple labels: 7102
Total rows: 43410
Percentage: 16.36%


In [6]:
# Step 1 - Map emotion indices to mood categories
emotion_to_mood = {
    0: 'happy',   # admiration
    1: 'happy',   # amusement
    2: 'angry',   # anger
    3: 'angry',   # annoyance
    4: 'happy',   # approval
    5: 'happy',   # caring
    6: 'neutral', # confusion
    7: 'neutral', # curiosity
    8: 'happy',   # desire
    9: 'sad',     # disappointment
    10: 'angry',  # disapproval
    11: 'angry',  # disgust
    12: 'sad',    # embarrassment
    13: 'happy',  # excitement
    14: 'sad',    # fear
    15: 'happy',  # gratitude
    16: 'sad',    # grief
    17: 'happy',  # joy
    18: 'happy',  # love
    19: 'sad',    # nervousness
    20: 'happy',  # optimism
    21: 'happy',  # pride
    22: 'neutral',# realization
    23: 'happy',  # relief
    24: 'sad',    # remorse
    25: 'sad',    # sadness
    26: 'neutral',# surprise
    27: 'neutral' # neutral
}

# Step 2 - Define priority order
mood_priority = {
    'angry': 0,    # highest priority
    'sad': 1,
    'happy': 2,
    'neutral': 3   # lowest priority
}

print("Mapping defined successfully!")

Mapping defined successfully!


In [7]:
# Step 3 - Priority function
def assign_mood(labels):
    # Convert each label number to mood category
    moods = [emotion_to_mood[label] for label in labels]
    
    # Apply priority rule - return highest priority mood
    return min(moods, key=lambda mood: mood_priority[mood])

# Test it
print(assign_mood([17, 2]))   # joy + anger → should return angry
print(assign_mood([27]))       # neutral → should return neutral
print(assign_mood([25, 17]))   # sadness + joy → should return sad

angry
neutral
sad


In [8]:
# Step 4 - Apply mood mapping to entire dataset
df_train['mood'] = df_train['labels'].apply(assign_mood)

# Check result
df_train[['text', 'labels', 'mood']].head(10)

,text,labels,mood
0,My favourite food is anything I didn't have to...,[27],neutral
1,"Now if he does off himself, everyone will thin...",[27],neutral
2,WHY THE FUCK IS BAYLESS ISOING,[2],angry
3,To make her feel threatened,[14],sad
4,Dirty Southern Wankers,[3],angry
5,OmG pEyToN iSn'T gOoD eNoUgH tO hElP uS iN tHe...,[26],neutral
6,Yes I heard abt the f bombs! That has to be wh...,[15],happy
7,We need more boards and to create a bit more s...,"[8, 20]",happy
8,Damn youtube and outrage drama is super lucrat...,[0],happy
9,It might be linked to the trust factor of your...,[27],neutral


In [9]:
# Check mood distribution
print(df_train['mood'].value_counts())
print()
print(df_train['mood'].value_counts(normalize=True).mul(100).round(2))

mood
neutral    16983
happy      16588
angry       6217
sad         3622
Name: count, dtype: int64

mood
neutral    39.12
happy      38.21
angry      14.32
sad         8.34
Name: proportion, dtype: float64


In [10]:
# Apply mood mapping to validation and test sets
df_val = dataset['validation'].to_pandas()
df_test = dataset['test'].to_pandas()

df_val['mood'] = df_val['labels'].apply(assign_mood)
df_test['mood'] = df_test['labels'].apply(assign_mood)

print(f"Train size: {len(df_train)}")
print(f"Validation size: {len(df_val)}")
print(f"Test size: {len(df_test)}")

Train size: 43410
Validation size: 5426
Test size: 5427


In [11]:
mkdir -p data/raw data/processed

In [12]:
# Save labeled datasets to processed folder
df_train.to_csv('data/processed/train_labeled.csv', index=False)
df_val.to_csv('data/processed/val_labeled.csv', index=False)
df_test.to_csv('data/processed/test_labeled.csv', index=False)

print("Datasets saved successfully!")
print("Location: data/processed/")

Datasets saved successfully!
Location: data/processed/
